In [1]:

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# Load the Karate Club network
G = nx.karate_club_graph()

# Remove edge weights for consistency with earlier experiments
G = nx.Graph(G)
for u, v in G.edges():
    G[u][v].clear()

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Nodes: 34
Edges: 78


In [2]:

def independent_cascade(G, seeds, p=0.1, seed=42):
    """
    Simulate information diffusion using Independent Cascade.

    Parameters
    ----------
    G : networkx.Graph
        Undirected, unweighted network.
    seeds : list
        Initially active nodes.
    p : float
        Activation probability for each attempt.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    activated : set
        All nodes activated by the end of the simulation.
    history : list
        Newly activated nodes at each diffusion step.
    """

    if not 0 <= p <= 1:
        raise ValueError("p must be between 0 and 1.")

    seeds = list(seeds)

    if not set(seeds).issubset(G.nodes):
        raise ValueError("All seed nodes must exist in G.")

    rng = np.random.default_rng(seed)

    activated = set(seeds)
    newly_active = set(seeds)
    history = [set(seeds)]

    while newly_active:
        next_active = set()

        for node in sorted(newly_active):
            for neighbour in sorted(G.neighbors(node)):

                if neighbour not in activated:
                    if rng.random() < p:
                        next_active.add(neighbour)

        next_active -= activated
        activated.update(next_active)

        if next_active:
            history.append(next_active)

        newly_active = next_active

    return activated, history


In [3]:

# Test Independent Cascade with node 0 as the seed

activated, history = independent_cascade(
    G,
    seeds=[0],
    p=0.1,
    seed=42
)

print("Starting seed:", [0])
print("Total activated nodes:", len(activated))
print("Percentage reached:", round(
    100 * len(activated) / G.number_of_nodes(), 2
), "%")

print("\nDiffusion history:")
for step, nodes in enumerate(history):
    print(f"Step {step}: {sorted(nodes)}")

# Basic correctness checks
assert len(activated) <= G.number_of_nodes()
assert set(history[0]) == {0}
assert set().union(*history) == activated

print("\nBasic checks passed!")


Starting seed: [0]
Total activated nodes: 3
Percentage reached: 8.82 %

Diffusion history:
Step 0: [0]
Step 1: [5]
Step 2: [10]

Basic checks passed!


In [4]:

# Step 3.3: Prepare centrality and community information

degree = nx.degree_centrality(G)

betweenness = nx.betweenness_centrality(
    G, normalized=True, weight=None
)

pagerank = nx.pagerank(
    G, alpha=0.85, weight=None
)

# Detect Louvain communities
communities = nx.community.louvain_communities(
    G, seed=42, weight=None
)

print("Number of communities:", len(communities))
print("Community sizes:", [len(c) for c in communities])


Number of communities: 3
Community sizes: [17, 4, 13]


In [5]:

def select_seeds(G, strategy, k=3, random_seed=42):
    """
    Select k initial seed nodes using a specified strategy.
    """

    if not 1 <= k <= G.number_of_nodes():
        raise ValueError("k must be between 1 and the number of nodes.")

    if strategy == "random":
        rng = np.random.default_rng(random_seed)
        return rng.choice(
            sorted(G.nodes()), size=k, replace=False
        ).tolist()

    elif strategy == "degree":
        return sorted(
            G.nodes(),
            key=lambda n: (-degree[n], n)
        )[:k]

    elif strategy == "betweenness":
        return sorted(
            G.nodes(),
            key=lambda n: (-betweenness[n], n)
        )[:k]

    elif strategy == "pagerank":
        return sorted(
            G.nodes(),
            key=lambda n: (-pagerank[n], n)
        )[:k]

    elif strategy == "community":
        # Rank nodes within each community by degree
        community_rankings = [
            sorted(c, key=lambda n: (-degree[n], n))
            for c in communities
        ]

        # Select one high-degree node from each community,
        # then repeat if more seeds are needed.
        seeds = []
        level = 0

        while len(seeds) < k:
            for ranking in community_rankings:
                if level < len(ranking):
                    seeds.append(ranking[level])

                    if len(seeds) == k:
                        break

            level += 1

        return seeds

    else:
        raise ValueError(f"Unknown strategy: {strategy}")


In [6]:

strategies = [
    "random",
    "degree",
    "betweenness",
    "pagerank",
    "community"
]

seed_results = []

for strategy in strategies:
    seeds = select_seeds(
        G,
        strategy=strategy,
        k=3,
        random_seed=42
    )

    seed_results.append({
        "Strategy": strategy,
        "Selected Seeds": seeds,
        "Communities Covered": len({
            i
            for i, c in enumerate(communities)
            if any(node in c for node in seeds)
        })
    })

seed_df = pd.DataFrame(seed_results)

display(seed_df)


,Strategy,Selected Seeds,Communities Covered
0,random,"[22, 2, 25]",3
1,degree,"[33, 0, 32]",2
2,betweenness,"[0, 33, 32]",2
3,pagerank,"[33, 0, 32]",2
4,community,"[0, 31, 33]",3


In [7]:

# Step 3.4: Repeated Independent Cascade simulations

n_simulations = 500
k = 3
p = 0.1
base_seed = 42

simulation_results = []

for strategy in strategies:

    for run in range(n_simulations):

        # Random strategy gets a new seed set each run
        # Other strategies use their fixed top-k nodes
        selected_seeds = select_seeds(
            G,
            strategy=strategy,
            k=k,
            random_seed=base_seed + run
        )

        activated, history = independent_cascade(
            G,
            seeds=selected_seeds,
            p=p,
            seed=10000 + run
        )

        simulation_results.append({
            "strategy": strategy,
            "run": run,
            "seed_nodes": selected_seeds,
            "activated_count": len(activated),
            "spread_percentage": (
                len(activated) / G.number_of_nodes() * 100
            ),
            "diffusion_steps": len(history) - 1
        })

simulation_df = pd.DataFrame(simulation_results)

# Summarise average information spread
summary_df = (
    simulation_df
    .groupby("strategy")
    .agg(
        mean_spread=("activated_count", "mean"),
        std_spread=("activated_count", "std"),
        min_spread=("activated_count", "min"),
        max_spread=("activated_count", "max"),
        mean_percentage=("spread_percentage", "mean"),
        mean_steps=("diffusion_steps", "mean")
    )
    .round(3)
    .sort_values("mean_spread", ascending=False)
)

display(summary_df)


,mean_spread,std_spread,min_spread,max_spread,mean_percentage,mean_steps
strategy,,,,,,
betweenness,8.038,2.373,3,18,23.641,1.658
degree,8.038,2.373,3,18,23.641,1.658
pagerank,8.038,2.373,3,18,23.641,1.658
community,7.490,2.503,3,19,22.029,1.738
random,5.330,2.525,3,16,15.676,1.374


In [8]:

# Step 3.5: Sensitivity analysis — seed budget

seed_budgets = [1, 2, 3, 4, 5]
n_simulations = 500
p = 0.1

budget_results = []

for k in seed_budgets:
    for strategy in strategies:

        spreads = []

        for run in range(n_simulations):

            selected_seeds = select_seeds(
                G,
                strategy=strategy,
                k=k,
                random_seed=42 + run
            )

            activated, _ = independent_cascade(
                G,
                seeds=selected_seeds,
                p=p,
                seed=10000 + run
            )

            spreads.append(len(activated))

        budget_results.append({
            "seed_budget": k,
            "strategy": strategy,
            "mean_spread": np.mean(spreads),
            "std_spread": np.std(spreads, ddof=1)
        })

budget_df = pd.DataFrame(budget_results)

display(
    budget_df.pivot(
        index="seed_budget",
        columns="strategy",
        values="mean_spread"
    ).round(3)
)


strategy,betweenness,community,degree,pagerank,random
seed_budget,,,,,
1,3.262,3.262,3.350,3.350,1.818
2,6.362,5.030,6.362,6.362,3.714
3,8.038,7.490,8.038,8.038,5.330
4,9.362,9.034,9.362,9.362,6.912
5,10.288,10.146,10.566,10.566,8.156


In [9]:

# Step 3.5B: Activation probability sensitivity

probabilities = [0.05, 0.10, 0.20, 0.30, 0.50]
n_simulations = 500
k = 3

probability_results = []

for p_value in probabilities:
    for strategy in strategies:

        spreads = []

        for run in range(n_simulations):

            selected_seeds = select_seeds(
                G,
                strategy=strategy,
                k=k,
                random_seed=42 + run
            )

            activated, _ = independent_cascade(
                G,
                seeds=selected_seeds,
                p=p_value,
                seed=10000 + run
            )

            spreads.append(len(activated))

        probability_results.append({
            "activation_probability": p_value,
            "strategy": strategy,
            "mean_spread": np.mean(spreads),
            "std_spread": np.std(spreads, ddof=1)
        })

probability_df = pd.DataFrame(probability_results)

display(
    probability_df.pivot(
        index="activation_probability",
        columns="strategy",
        values="mean_spread"
    ).round(3)
)


strategy,betweenness,community,degree,pagerank,random
activation_probability,,,,,
0.05,5.336,4.992,5.336,5.336,3.804
0.10,8.038,7.490,8.038,8.038,5.330
0.20,14.420,14.218,14.420,14.420,10.894
0.30,20.626,20.764,20.626,20.626,19.062
0.50,28.646,28.758,28.646,28.646,29.002
